# 09 | Acquisition-aware training and specimen-balanced sampling

## Swin-T robustness ablation on unseen steel specimens

This experiment examines whether changes to augmentation and training-sample weighting affect Swin-T classification of Bainite and Martensite on a fixed specimen-disjoint partition. Four conditions are evaluated: reference training, acquisition-aware augmentation, specimen-balanced sampling, and their combination.

The earlier Swin-T model achieved a macro-F1 of **0.7932** on its specimen-held-out test partition. That value provides historical context rather than a matched control for all training choices examined here.

**Research question.** Do acquisition-oriented image transformations or more equal specimen representation improve performance on the selected unseen-specimen partition? Each condition uses validation-guided checkpoint selection, followed by held-out test evaluation.

## 1. Configuration

Define the dataset locations, Swin-T checkpoint and output directories, optimization settings, random seed and computational parameters. This notebook writes its experiment artifacts under the existing `08_swin` model and results directories.

In [1]:
from pathlib import Path
PROCESSED_DIR=Path(r"../data/processed")
MODEL_DIR=Path(r"../models/08_swin"); MODEL_DIR.mkdir(parents=True,exist_ok=True)
RESULT_DIR=Path(r"../results/08_swin"); RESULT_DIR.mkdir(parents=True,exist_ok=True)

SEED=42
IMAGE_SIZE=224
BATCH_SIZE=16
NUM_WORKERS=0
HEAD_EPOCHS=4
HEAD_LR=1e-3
FINETUNE_EPOCHS=15
FINETUNE_LR=2e-5
PATIENCE=4
MIN_DELTA=1e-4
WEIGHT_DECAY=1e-4
RESUME=True

print("Maximum epochs per experiment:",HEAD_EPOCHS+FINETUNE_EPOCHS)
print("Resume enabled:",RESUME)

Maximum epochs per experiment: 19
Resume enabled: True


## 2. Dependencies and reproducibility

Load the image-processing, modelling and evaluation libraries and establish deterministic settings where supported. Hardware- and implementation-level differences may still affect exact reproducibility.

In [2]:
import random,time,warnings,numpy as np,pandas as pd,matplotlib.pyplot as plt
from PIL import Image
import torch,torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms
from torchvision.models import swin_t,Swin_T_Weights
from sklearn.metrics import accuracy_score,balanced_accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix
from IPython.display import display
warnings.filterwarnings("ignore")

def set_seed(seed=42):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:",torch.__version__,"| CUDA:",torch.cuda.is_available(),"| Device:",device)
if device.type=="cpu": print("CPU threads:",torch.get_num_threads())

PyTorch: 2.14.0+cpu | CUDA: False | Device: cpu
CPU threads: 4


## 3. Evaluation manifests

Load the stratified and specimen-grouped manifests prepared previously. The robustness ablations below are performed on the specimen-grouped partition.

In [3]:
standard_path=PROCESSED_DIR/"split_baseline_stratified.csv"
sample_path=PROCESSED_DIR/"split_sample_grouped.csv"
assert standard_path.exists(),f"Missing {standard_path}"
assert sample_path.exists(),f"Missing {sample_path}"
experiments={"standard_stratified":pd.read_csv(standard_path),"sample_grouped":pd.read_csv(sample_path)}
print("Standard:",standard_path)
print("Sample grouped:",sample_path)

Standard: ..\data\processed\split_baseline_stratified.csv
Sample grouped: ..\data\processed\split_sample_grouped.csv


## 4. Partition audit

Verify the required columns, image availability and class representation across train, validation and test subsets. The specimen-disjoint split remains the primary basis for evaluating generalization to new physical specimens.

In [4]:
rows=[]
for name,df in experiments.items():
    required={"image_path","Phase","label","split"}
    assert not (required-set(df.columns)),f"{name}: missing {required-set(df.columns)}"
    for split in ["train","val","test"]:
        d=df[df.split==split]
        assert len(d)>0 and d.Phase.nunique()==2
        assert all(Path(p).exists() for p in d.image_path)
        rows.append({"experiment":name,"split":split,"n_images":len(d),
                     "bainite":(d.Phase=="Bainite").sum(),"martensite":(d.Phase=="Martensite").sum(),
                     "n_samples":d.SampleIDHarm.nunique() if "SampleIDHarm" in d else np.nan})
display(pd.DataFrame(rows))

,experiment,split,n_images,bainite,martensite,n_samples
0,standard_stratified,train,3705,1729,1976,20
1,standard_stratified,val,794,370,424,20
2,standard_stratified,test,794,371,423,20
3,sample_grouped,train,3811,1786,2025,14
4,sample_grouped,val,722,342,380,3
5,sample_grouped,test,760,342,418,3


## 5. Input preprocessing

Swin-T uses 224 × 224 RGB inputs and ImageNet channel normalization. The reference training transform incorporates flips and mild rotation; validation and test images undergo deterministic resizing and normalization.

The acquisition-aware condition, defined below, additionally perturbs translation, scale, brightness, contrast and blur. These transformations approximate selected sources of acquisition variability but do not reproduce the full physics of microscopy.

In [5]:
weights=Swin_T_Weights.DEFAULT
MEAN=[.485,.456,.406];STD=[.229,.224,.225]
train_transform=transforms.Compose([
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),transforms.RandomRotation(10),
    transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
eval_transform=transforms.Compose([
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])

class MicrostructureDataset(Dataset):
    def __init__(self,df,transform): self.df=df.reset_index(drop=True).copy();self.transform=transform
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        with Image.open(r.image_path) as im: x=self.transform(im.convert("RGB"))
        return x,torch.tensor(int(r.label),dtype=torch.long),i

def make_loaders(df):
    parts={s:df[df.split==s].copy() for s in ["train","val","test"]}
    ds={s:MicrostructureDataset(parts[s],train_transform if s=="train" else eval_transform) for s in parts}
    common=dict(batch_size=BATCH_SIZE,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available())
    g=torch.Generator().manual_seed(SEED)
    loaders={"train":DataLoader(ds["train"],shuffle=True,generator=g,**common),
             "val":DataLoader(ds["val"],shuffle=False,**common),
             "test":DataLoader(ds["test"],shuffle=False,**common)}
    return parts,ds,loaders

## 6. Swin-T architecture and fine-tuning

Initialize Swin-T with pretrained weights and replace its classification head for the two phase labels. Training proceeds in two stages: optimization of the classifier head, followed by partial fine-tuning of the final transformer stage, normalization layer and head.

In [6]:
def build_model():
    m=swin_t(weights=weights)
    m.head=nn.Linear(m.head.in_features,2)
    return m

def freeze_backbone(m):
    for p in m.parameters(): p.requires_grad=False
    for p in m.head.parameters(): p.requires_grad=True

def unfreeze_final_stage(m):
    for p in m.parameters(): p.requires_grad=False
    for p in m.features[7].parameters(): p.requires_grad=True
    for p in m.norm.parameters(): p.requires_grad=True
    for p in m.head.parameters(): p.requires_grad=True

def report(m):
    total=sum(p.numel() for p in m.parameters());train=sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"Total parameters: {total:,}\nTrainable parameters: {train:,}\nTrainable fraction: {100*train/total:.2f}%")

## 7. Evaluation metrics

Report accuracy, balanced accuracy, macro-F1 and ROC-AUC. Macro-F1 is used for validation checkpoint selection and primary comparison across interventions.

In [7]:
def metrics(y,p,prob):
    return {"accuracy":accuracy_score(y,p),"balanced_accuracy":balanced_accuracy_score(y,p),
            "precision_macro":precision_score(y,p,average="macro",zero_division=0),
            "recall_macro":recall_score(y,p,average="macro",zero_division=0),
            "f1_macro":f1_score(y,p,average="macro",zero_division=0),"roc_auc":roc_auc_score(y,prob)}

@torch.inference_mode()
def evaluate(m,loader,criterion):
    m.eval();loss_sum=0;y=[];p=[];prob=[];idxs=[]
    for x,t,idx in loader:
        x,t=x.to(device),t.to(device);z=m(x);loss=criterion(z,t)
        pr=torch.softmax(z,1)[:,1];pred=z.argmax(1)
        loss_sum+=loss.item()*x.size(0);y.extend(t.cpu().numpy());p.extend(pred.cpu().numpy());prob.extend(pr.cpu().numpy());idxs.extend(idx.numpy())
    out=metrics(y,p,prob);out["loss"]=loss_sum/len(loader.dataset)
    return out,np.asarray(y),np.asarray(p),np.asarray(prob),np.asarray(idxs)

## 8. Checkpointing and training history

Persist the highest-validation-F1 checkpoint, latest optimizer/training state and epoch-level history. This supports recovery from interruptions without using the held-out test set for checkpoint selection.

In [8]:
def paths(name):
    return {"best":MODEL_DIR/f"{name}_swin_t_best.pt","latest":MODEL_DIR/f"{name}_swin_t_latest.pt",
            "history":RESULT_DIR/f"{name}_history.csv"}

def train_stage(m,ds,loaders,name,stage,max_epochs,lr,offset,start=1,best=-np.inf,stale=0,resume=None):
    ps=paths(name);criterion=nn.CrossEntropyLoss()
    opt=torch.optim.AdamW([p for p in m.parameters() if p.requires_grad],lr=lr,weight_decay=WEIGHT_DECAY)
    sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",factor=.5,patience=2,min_lr=1e-7)
    if resume:
        try:
            opt.load_state_dict(resume["optimizer_state_dict"]);sched.load_state_dict(resume["scheduler_state_dict"])
            print("✓ optimizer/scheduler restored")
        except Exception as e: print("Fresh optimizer used:",e)
    hist=pd.read_csv(ps["history"]).to_dict("records") if ps["history"].exists() else []
    for local in range(start,max_epochs+1):
        epoch=offset+local;m.train();ls=0;y=[];pred=[];prob=[]
        for x,t,_ in loaders["train"]:
            x,t=x.to(device),t.to(device);opt.zero_grad(set_to_none=True);z=m(x);loss=criterion(z,t);loss.backward();opt.step()
            pr=torch.softmax(z,1)[:,1];pd_=z.argmax(1);ls+=loss.item()*x.size(0)
            y.extend(t.detach().cpu().numpy());pred.extend(pd_.detach().cpu().numpy());prob.extend(pr.detach().cpu().numpy())
        tr=metrics(y,pred,prob);tr["loss"]=ls/len(ds["train"])
        va,*_=evaluate(m,loaders["val"],criterion);sched.step(va["f1_macro"]);cur=opt.param_groups[0]["lr"]
        rec={"epoch":epoch,"stage":stage,"lr":cur,**{f"train_{k}":v for k,v in tr.items()},**{f"val_{k}":v for k,v in va.items()}}
        hist=[r for r in hist if not(int(r["epoch"])==epoch and str(r["stage"])==stage)]+[rec]
        pd.DataFrame(hist).sort_values("epoch").to_csv(ps["history"],index=False)
        improved=va["f1_macro"]>best+MIN_DELTA
        if improved:
            best=va["f1_macro"];stale=0
            torch.save({"model_state_dict":m.state_dict(),"architecture":"swin_t","image_size":IMAGE_SIZE,
                        "label_map":{"Bainite":0,"Martensite":1},"best_val_macro_f1":float(best),
                        "stage":stage,"best_epoch":epoch,"seed":SEED},ps["best"])
            mark=" ← BEST SAVED"
        else: stale+=1;mark=""
        torch.save({"model_state_dict":m.state_dict(),"optimizer_state_dict":opt.state_dict(),
                    "scheduler_state_dict":sched.state_dict(),"stage":stage,"local_epoch":local,
                    "global_epoch":epoch,"best_f1":float(best),"stale":stale},ps["latest"])
        print(f"Epoch {epoch:02d} | {stage:9s} | train F1 {tr['f1_macro']:.4f} | val F1 {va['f1_macro']:.4f} | val AUC {va['roc_auc']:.4f} | lr {cur:.1e}{mark}")
        if stale>=PATIENCE:
            print(f"Early stopping {stage}.");break
    return best

## 9. Experiment execution

The training routine restores an available checkpoint when resume is enabled, applies the two-stage schedule and evaluates the selected model on the test subset. Prediction-level outputs are saved for inspection.

In [9]:
def load_best(m,p):
    ck=torch.load(p,map_location=device,weights_only=False);m.load_state_dict(ck["model_state_dict"]);return ck

def run_experiment(name,df):
    print("\n"+"="*88+f"\nSwin-T: {name}\n"+"="*88)
    set_seed(SEED);parts,ds,loaders=make_loaders(df);ps=paths(name);m=build_model().to(device);start_time=time.time()
    resume=torch.load(ps["latest"],map_location=device,weights_only=False) if RESUME and ps["latest"].exists() else None
    best=-np.inf
    if resume: print(f"Resume found: {resume['stage']} epoch {resume['global_epoch']} | best {resume['best_f1']:.4f}")
    if resume and resume["stage"]=="fine_tune":
        best=resume["best_f1"]
    else:
        freeze_backbone(m);print("\nSTAGE 1 — CLASSIFIER HEAD");report(m)
        if resume and resume["stage"]=="head":
            m.load_state_dict(resume["model_state_dict"]);best=resume["best_f1"];start=resume["local_epoch"]+1
            if start<=HEAD_EPOCHS: best=train_stage(m,ds,loaders,name,"head",HEAD_EPOCHS,HEAD_LR,0,start,best,resume["stale"],resume)
        else:
            best=train_stage(m,ds,loaders,name,"head",HEAD_EPOCHS,HEAD_LR,0)
    if resume and resume["stage"]=="fine_tune":
        unfreeze_final_stage(m);m.load_state_dict(resume["model_state_dict"]);start=resume["local_epoch"]+1
        print("\nSTAGE 2 — FINAL SWIN STAGE (RESUMED)");report(m)
        if start<=FINETUNE_EPOCHS: best=train_stage(m,ds,loaders,name,"fine_tune",FINETUNE_EPOCHS,FINETUNE_LR,HEAD_EPOCHS,start,best,resume["stale"],resume)
    else:
        assert ps["best"].exists();load_best(m,ps["best"]);unfreeze_final_stage(m)
        print("\nSTAGE 2 — FINAL SWIN STAGE");report(m)
        best=train_stage(m,ds,loaders,name,"fine_tune",FINETUNE_EPOCHS,FINETUNE_LR,HEAD_EPOCHS,1,best)
    ck=load_best(m,ps["best"]);criterion=nn.CrossEntropyLoss()
    tm,y,p,prob,idx=evaluate(m,loaders["test"],criterion);mins=(time.time()-start_time)/60
    out=parts["test"].reset_index(drop=True).iloc[idx].reset_index(drop=True).copy()
    out["true_label"]=y;out["predicted_label"]=p;out["p_martensite"]=prob
    out.to_csv(RESULT_DIR/f"{name}_test_predictions.csv",index=False)
    print("\nTEST RESULTS — BEST VALIDATION CHECKPOINT")
    print("Best validation epoch:",ck["best_epoch"]);print("Best validation F1:",f"{ck['best_val_macro_f1']:.4f}")
    for k,v in tm.items():print(f"{k:20s}: {v:.4f}")
    print(f"Session runtime: {mins:.1f} min\nBest checkpoint: {ps['best']}")
    return {"name":name,"metrics":tm,"y":y,"pred":p,"prob":prob,"minutes":mins,"best_epoch":ck["best_epoch"]}

## 10. Reference training implementation

The training routine is defined above. The following section specifies the shorter intervention-screening schedule and invokes the actual ablation conditions.

## 11. Robustness intervention settings

The acquisition-aware transform adds stronger rotation, translation, scaling, brightness and contrast jitter, and occasional Gaussian blur to the reference training preprocessing. These changes are applied only to training images.

The ablation uses a shorter fine-tuning budget than the earlier architecture comparison, so direct differences from that historical benchmark should not be attributed solely to augmentation.

In [10]:
# Shorter intervention screen than Notebook 08.
HEAD_EPOCHS = 4
FINETUNE_EPOCHS = 12
PATIENCE = 4
RESUME = True

sample_df = pd.read_csv(PROCESSED_DIR / "split_sample_grouped.csv")

# Stronger acquisition-oriented augmentation.
robust_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.05,0.05), scale=(0.90,1.10)),
    transforms.ColorJitter(brightness=0.20, contrast=0.20),
    transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1,1.0))], p=0.20),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

print("Sample-grouped rows:", len(sample_df))
print("Notebook 08 Swin unseen-specimen F1 benchmark: 0.7932")


Sample-grouped rows: 5293
Notebook 08 Swin unseen-specimen F1 benchmark: 0.7932


## 12. Specimen-balanced sampling

With image-uniform sampling, specimens containing more images contribute more training examples. The weighted sampler assigns each training image a weight inversely proportional to the number of images from its physical specimen, approximately equalizing each specimen's expected sampling contribution.

Sampling is performed with replacement; it does not change validation or test composition.

In [11]:
from torch.utils.data import WeightedRandomSampler

def make_domain_loaders(df, train_tf, sample_balanced=False):
    parts={s:df[df["split"]==s].copy() for s in ["train","val","test"]}
    datasets={
        "train":MicrostructureDataset(parts["train"],train_tf),
        "val":MicrostructureDataset(parts["val"],eval_transform),
        "test":MicrostructureDataset(parts["test"],eval_transform)
    }
    common=dict(batch_size=BATCH_SIZE,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available())

    if sample_balanced:
        tr=parts["train"].reset_index(drop=True)
        counts=tr["SampleIDHarm"].value_counts()
        w=tr["SampleIDHarm"].map(lambda x:1.0/counts[x]).to_numpy()
        sampler=WeightedRandomSampler(torch.as_tensor(w,dtype=torch.double),
                                      num_samples=len(w),replacement=True,
                                      generator=torch.Generator().manual_seed(SEED))
        train_loader=DataLoader(datasets["train"],sampler=sampler,shuffle=False,**common)
    else:
        train_loader=DataLoader(datasets["train"],shuffle=True,
                                generator=torch.Generator().manual_seed(SEED),**common)

    loaders={"train":train_loader,
             "val":DataLoader(datasets["val"],shuffle=False,**common),
             "test":DataLoader(datasets["test"],shuffle=False,**common)}
    return parts,datasets,loaders


## 13. Intervention runner

Apply the selected training transform and sampling strategy while keeping the common model architecture and evaluation procedure.

In [12]:
def run_domain_intervention(name, train_tf, sample_balanced):
    global make_loaders, train_transform

    old_make_loaders = make_loaders
    old_train_transform = train_transform

    train_transform = train_tf
    make_loaders = lambda dataframe: make_domain_loaders(
        dataframe, train_tf, sample_balanced=sample_balanced
    )

    # Separate filenames for each ablation.
    result = run_experiment(name, sample_df)

    make_loaders = old_make_loaders
    train_transform = old_train_transform
    return result


## 14. Ablation conditions

The four conditions isolate the effects of acquisition-aware augmentation and specimen-balanced sampling, individually and jointly, on the same specimen-held-out split.

The `nb09_reference` condition is the internal control under this notebook's training schedule. Test-set comparisons describe this specific partition and do not establish robustness across alternative held-out specimen selections.

In [13]:
interventions = {
    "nb09_reference": (train_transform, False),
    "nb09_acquisition_aug": (robust_transform, False),
    "nb09_sample_balanced": (train_transform, True),
    "nb09_combined": (robust_transform, True),
}

ablation_results={}
for name,(tf,balanced) in interventions.items():
    ablation_results[name]=run_domain_intervention(name,tf,balanced)

print("\n✓ All four domain-robust interventions completed.")



Swin-T: nb09_reference

STAGE 1 — CLASSIFIER HEAD
Total parameters: 27,520,892
Trainable parameters: 1,538
Trainable fraction: 0.01%
Epoch 01 | head      | train F1 0.8161 | val F1 0.6744 | val AUC 0.8332 | lr 1.0e-03 ← BEST SAVED
Epoch 02 | head      | train F1 0.8494 | val F1 0.7472 | val AUC 0.8503 | lr 1.0e-03 ← BEST SAVED
Epoch 03 | head      | train F1 0.8626 | val F1 0.7590 | val AUC 0.8407 | lr 1.0e-03 ← BEST SAVED
Epoch 04 | head      | train F1 0.8709 | val F1 0.6959 | val AUC 0.8336 | lr 1.0e-03

STAGE 2 — FINAL SWIN STAGE
Total parameters: 27,520,892
Trainable parameters: 14,186,930
Trainable fraction: 51.55%
Epoch 05 | fine_tune | train F1 0.8671 | val F1 0.7452 | val AUC 0.8563 | lr 2.0e-05
Epoch 06 | fine_tune | train F1 0.8870 | val F1 0.7361 | val AUC 0.8636 | lr 2.0e-05
Epoch 07 | fine_tune | train F1 0.8988 | val F1 0.7069 | val AUC 0.8646 | lr 2.0e-05
Epoch 08 | fine_tune | train F1 0.9112 | val F1 0.7320 | val AUC 0.8721 | lr 1.0e-05
Early stopping fine_tune.

TES

## 15. Test-set comparison

Aggregate the four interventions' test metrics, best validation epochs and runtimes. Preserve the complete table rather than interpreting only the highest-scoring configuration.

In [14]:
rows=[]
for name,r in ablation_results.items():
    m=r["metrics"]
    rows.append({
        "Intervention":name,
        "Accuracy":m["accuracy"],
        "Balanced Accuracy":m["balanced_accuracy"],
        "Macro-F1":m["f1_macro"],
        "ROC-AUC":m["roc_auc"],
        "Best val epoch":r["best_epoch"],
        "Runtime (min)":r["minutes"],
    })
summary=pd.DataFrame(rows).sort_values("Macro-F1",ascending=False)
display(summary.style.format({
    "Accuracy":"{:.4f}","Balanced Accuracy":"{:.4f}",
    "Macro-F1":"{:.4f}","ROC-AUC":"{:.4f}","Runtime (min)":"{:.1f}"
}))
summary.to_csv(RESULT_DIR/"nb09_domain_robust_ablation.csv",index=False)


,Intervention,Accuracy,Balanced Accuracy,Macro-F1,ROC-AUC,Best val epoch,Runtime (min)
1,nb09_acquisition_aug,0.8118,0.8048,0.8074,0.8924,10,114.3
0,nb09_reference,0.7974,0.7911,0.7932,0.8728,3,63.0
2,nb09_sample_balanced,0.7921,0.7889,0.7895,0.8766,7,87.0
3,nb09_combined,0.7855,0.7835,0.7834,0.8685,12,130.4


## 16. Historical architecture benchmarks

Place the ablation results alongside earlier CNN, ResNet18, ViT-B/16 and Swin-T unseen-specimen results. Because training configurations differ across experiments, this comparison is descriptive rather than a controlled estimate of the effect of a single intervention.

In [15]:
prior=pd.DataFrame([
    ["Custom CNN",0.6405,0.7094],
    ["ResNet18",0.6702,0.7747],
    ["ViT-B/16",0.7497,0.8190],
    ["Swin-T (Notebook 08)",0.7932,0.8728],
],columns=["Method","Macro-F1","ROC-AUC"])

current=summary[["Intervention","Macro-F1","ROC-AUC"]].copy()
current["Method"]="Swin-T + "+current["Intervention"]
comparison=pd.concat([prior,current[["Method","Macro-F1","ROC-AUC"]]],ignore_index=True)
display(comparison.style.format({"Macro-F1":"{:.4f}","ROC-AUC":"{:.4f}"}))
comparison.to_csv(RESULT_DIR/"nb09_all_unseen_specimen_comparison.csv",index=False)


,Method,Macro-F1,ROC-AUC
0,Custom CNN,0.6405,0.7094
1,ResNet18,0.6702,0.7747
2,ViT-B/16,0.7497,0.8190
3,Swin-T (Notebook 08),0.7932,0.8728
4,Swin-T + nb09_acquisition_aug,0.8074,0.8924
5,Swin-T + nb09_reference,0.7932,0.8728
6,Swin-T + nb09_sample_balanced,0.7895,0.8766
7,Swin-T + nb09_combined,0.7834,0.8685


## 17. Experiment summary

Record all four test results and the highest-scoring condition on the evaluated specimen partition. The ranking represents the observed outcome for this split only.

In [16]:
best=summary.iloc[0]
print("="*88)
print("DOMAIN-ROBUST TRAINING SUMMARY")
print("="*88)
for _,r in summary.iterrows():
    print(f"\n{r['Intervention']}")
    print(f"  Accuracy          : {r['Accuracy']:.4f}")
    print(f"  Balanced Accuracy : {r['Balanced Accuracy']:.4f}")
    print(f"  Macro-F1          : {r['Macro-F1']:.4f}")
    print(f"  ROC-AUC           : {r['ROC-AUC']:.4f}")
    print(f"  Best val epoch    : {int(r['Best val epoch'])}")
    print(f"  Runtime           : {r['Runtime (min)']:.1f} min")

print("\nPREVIOUS UNSEEN-SPECIMEN BENCHMARKS")
print("  Custom CNN        : 0.6405")
print("  ResNet18          : 0.6702")
print("  ViT-B/16          : 0.7497")
print("  Swin-T NB08       : 0.7932")

print("\nBEST NB09 INTERVENTION")
print(f"  Strategy          : {best['Intervention']}")
print(f"  Macro-F1          : {best['Macro-F1']:.4f}")
print(f"  Delta vs NB08     : {best['Macro-F1']-0.7932:+.4f}")
print("\nNEXT: Notebook 10 — grouped robustness/statistical validation")
print("="*88)


DOMAIN-ROBUST TRAINING SUMMARY

nb09_acquisition_aug
  Accuracy          : 0.8118
  Balanced Accuracy : 0.8048
  Macro-F1          : 0.8074
  ROC-AUC           : 0.8924
  Best val epoch    : 10
  Runtime           : 114.3 min

nb09_reference
  Accuracy          : 0.7974
  Balanced Accuracy : 0.7911
  Macro-F1          : 0.7932
  ROC-AUC           : 0.8728
  Best val epoch    : 3
  Runtime           : 63.0 min

nb09_sample_balanced
  Accuracy          : 0.7921
  Balanced Accuracy : 0.7889
  Macro-F1          : 0.7895
  ROC-AUC           : 0.8766
  Best val epoch    : 7
  Runtime           : 87.0 min

nb09_combined
  Accuracy          : 0.7855
  Balanced Accuracy : 0.7835
  Macro-F1          : 0.7834
  ROC-AUC           : 0.8685
  Best val epoch    : 12
  Runtime           : 130.4 min

PREVIOUS UNSEEN-SPECIMEN BENCHMARKS
  Custom CNN        : 0.6405
  ResNet18          : 0.6702
  ViT-B/16          : 0.7497
  Swin-T NB08       : 0.7932

BEST NB09 INTERVENTION
  Strategy          : nb09_ac

## Interpretation and limitations

On the original specimen-disjoint partition, acquisition-aware augmentation improved the Swin-T macro-F1 from **0.7932** in the earlier benchmark to **0.8074** in this intervention screen. That comparison is not sufficient to establish a reproducible improvement, particularly because the training budgets and experimental configurations are not identical.

Subsequent repeated specimen-grouped validation (NB10) found **no consistent advantage from acquisition-aware augmentation**: mean macro-F1 was **0.7703 ± 0.0643** with augmentation versus **0.7769 ± 0.0826** for Swin-T without it, across five partitions. The paired mean difference was **−0.0066**, with a 95% confidence interval of **[−0.0459, +0.0327]**.

Accordingly, this notebook documents an exploratory, split-specific ablation—not evidence that the selected intervention universally improves microscopy-domain robustness. Robustness claims must be grounded in the repeated grouped results rather than the maximum score obtained on one partition.